In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from models import *

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant_4bit"
model = VGG16_quant()
model.features[0].act_alpha.data.fill_(1.0)
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [ ]:
lr = 4e-2
weight_decay = 1e-4
epochs = 150
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    #adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    scheduler.step()

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

In [2]:
PATH = "result/VGG16_quant_4bit/model_best.pth.tar"
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8945/10000 (89%)



In [7]:
from models.quant_layer import QuantConv2d, act_quantization

class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []  
        
######### Save inputs from selected layer ##########
save_output = SaveOutput()
i = 0

for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        print(i,"-th layer prehooked")
        layer.register_forward_pre_hook(save_output)             
####################################################

dataiter = iter(testloader)
images, labels = next(dataiter)
images = images.to(device)
out = model(images)

3 -th layer prehooked
7 -th layer prehooked
12 -th layer prehooked
16 -th layer prehooked
21 -th layer prehooked
25 -th layer prehooked
29 -th layer prehooked
34 -th layer prehooked
38 -th layer prehooked
42 -th layer prehooked
47 -th layer prehooked
51 -th layer prehooked
55 -th layer prehooked


In [31]:
weight_q = model.features[3].weight_q
w_alpha = model.features[3].weight_quant.wgt_alpha
w_bit = 4

weight_int = weight_q / (w_alpha / (2**(w_bit-1)-1))
print(weight_int.size())
print(weight_int)

torch.Size([64, 64, 3, 3])
tensor([[[[ 1.0000,  0.0000,  2.0000],
          [ 2.0000,  3.0000,  1.0000],
          [ 1.0000,  2.0000,  1.0000]],

         [[ 1.0000,  1.0000, -0.0000],
          [ 1.0000,  1.0000,  1.0000],
          [ 1.0000, -0.0000, -1.0000]],

         [[-1.0000,  1.0000,  1.0000],
          [ 1.0000,  2.0000,  1.0000],
          [-0.0000,  1.0000, -0.0000]],

         ...,

         [[ 2.0000,  1.0000,  1.0000],
          [ 2.0000,  2.0000,  2.0000],
          [ 1.0000,  0.0000,  1.0000]],

         [[ 2.0000,  1.0000,  2.0000],
          [ 2.0000,  3.0000,  3.0000],
          [ 2.0000,  1.0000,  2.0000]],

         [[ 1.0000,  3.0000,  2.0000],
          [ 1.0000,  2.0000,  1.0000],
          [ 1.0000,  1.0000,  1.0000]]],


        [[[ 6.0000, -7.0000,  2.0000],
          [-7.0000,  7.0000, -6.0000],
          [-1.0000, -3.0000,  0.0000]],

         [[-2.0000, -2.0000, -4.0000],
          [ 1.0000, -0.0000, -2.0000],
          [ 3.0000,  2.0000,  2.0000]],

    

In [32]:
act = save_output.outputs[1][0]
act_alpha  = model.features[3].act_alpha
act_bit = 4
act_quant_fn = act_quantization(act_bit)

act_q = act_quant_fn(act, act_alpha)

act_int = act_q / (act_alpha / (2**act_bit-1))
print(act_int.size())
print(act_int)

torch.Size([128, 64, 32, 32])
tensor([[[[ 4.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [10.0000,  3.0000,  0.0000,  ...,  3.0000,  1.0000,  0.0000],
          [11.0000,  2.0000,  1.0000,  ...,  5.0000,  2.0000,  0.0000],
          ...,
          [ 0.0000,  5.0000,  3.0000,  ..., 11.0000,  0.0000, 10.0000],
          [ 0.0000,  6.0000,  2.0000,  ...,  8.0000,  4.0000,  5.0000],
          [ 0.0000,  1.0000,  0.0000,  ...,  0.0000,  1.0000,  0.0000]],

         [[ 1.0000,  1.0000,  1.0000,  ...,  2.0000,  2.0000,  1.0000],
          [ 1.0000,  2.0000,  2.0000,  ...,  3.0000,  2.0000,  1.0000],
          [ 2.0000,  2.0000,  2.0000,  ...,  2.0000,  2.0000,  1.0000],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],

         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],


In [33]:
conv_int = torch.nn.Conv2d(in_channels = 64, out_channels=64, kernel_size = 3, padding=1)
conv_int.weight = torch.nn.parameter.Parameter(weight_int)
conv_int.bias = model.features[3].bias
output_int = conv_int(act_int)
output_recovered = output_int * (act_alpha / (2**act_bit-1)) * (w_alpha / (2**(w_bit-1)-1))
print(output_recovered)

tensor([[[[ 1.3019e+01,  6.0215e+00,  4.0686e+00,  ...,  1.0660e+01,
            9.6832e+00, -5.6960e-01],
          [ 3.4908e+01,  3.8489e+01,  3.0026e+01,  ...,  4.0116e+01,
            3.1979e+01,  1.2613e+01],
          [ 3.5803e+01,  4.1093e+01,  3.4990e+01,  ...,  3.8163e+01,
            3.3200e+01,  1.7902e+01],
          ...,
          [ 4.6219e+01,  6.7782e+01,  6.5016e+01,  ...,  7.8442e+01,
            6.0459e+01,  4.3290e+01],
          [ 3.8000e+01,  4.8416e+01,  4.8823e+01,  ...,  5.4763e+01,
            4.6951e+01,  3.6698e+01],
          [ 2.7422e+01,  3.5478e+01,  4.1011e+01,  ...,  3.6861e+01,
            3.6048e+01,  2.6608e+01]],

         [[-1.0497e+01, -5.8587e+00, -6.5097e-01,  ..., -3.1735e+00,
           -1.1962e+01, -1.7495e+01],
          [-1.4728e+01, -7.4862e+00, -1.8064e+01,  ..., -1.9773e+01,
           -1.9773e+01,  2.1970e+00],
          [-2.3598e+01, -3.0596e+01, -4.3127e+00,  ..., -2.9701e+01,
           -3.2793e+01, -1.5054e+01],
          ...,
     

In [34]:
conv_ref = torch.nn.Conv2d(in_channels = 64, out_channels=64, kernel_size = 3, padding=1)
conv_ref.weight = model.features[3].weight_q
conv_ref.bias = model.features[3].bias
output_ref = conv_ref(act)
print(output_ref)

tensor([[[[ 1.3416e+01,  6.5831e+00,  4.3029e+00,  ...,  1.0646e+01,
            9.8327e+00, -1.6347e-01],
          [ 3.5628e+01,  3.9854e+01,  3.0939e+01,  ...,  4.0719e+01,
            3.2455e+01,  1.2796e+01],
          [ 3.6407e+01,  4.2004e+01,  3.5296e+01,  ...,  3.8758e+01,
            3.3684e+01,  1.8434e+01],
          ...,
          [ 4.6897e+01,  6.7742e+01,  6.4573e+01,  ...,  7.8701e+01,
            6.0714e+01,  4.3695e+01],
          [ 3.8020e+01,  4.8018e+01,  4.8282e+01,  ...,  5.4927e+01,
            4.7076e+01,  3.7536e+01],
          [ 2.8030e+01,  3.5243e+01,  4.0885e+01,  ...,  3.7252e+01,
            3.6362e+01,  2.7113e+01]],

         [[-1.0551e+01, -5.1116e+00,  8.0350e-01,  ..., -4.9053e+00,
           -1.2475e+01, -1.8880e+01],
          [-1.5704e+01, -5.6331e+00, -1.9023e+01,  ..., -1.8194e+01,
           -1.9286e+01,  1.6779e+00],
          [-2.4510e+01, -2.9693e+01, -4.7770e+00,  ..., -3.0008e+01,
           -3.1563e+01, -1.5582e+01],
          ...,
     

In [35]:
# act_int.size = torch.Size([128, 64, 32, 32])  <- batch_size, input_ch, ni, nj
a_int = act_int[0,:,:,:]  # pick only one input out of batch
# a_int.size() = [64, 32, 32]

# conv_int.weight.size() = torch.Size([64, 64, 3, 3])  <- output_ch, input_ch, ki, kj
w_int = torch.reshape(weight_int, (weight_int.size(0), weight_int.size(1), -1))  # merge ki, kj index to kij
# w_int.weight.size() = torch.Size([64, 64, 9])
                      
padding = 1
stride = 1
array_size = 16 # row and column number
tile = array_size
n_t = int(w_int.size(0) / array_size)

nig = range(a_int.size(1))  ## ni group [0,1,...31]
njg = range(a_int.size(2))  ## nj group
 
icg = range(int(w_int.size(1)))  ## input channel [0,...63]
ocg = range(int(w_int.size(0)))  ## output channel


kijg = range(w_int.size(2)) # [0, .. 8]
ki_dim = int(math.sqrt(w_int.size(2)))  ## Kernel's 1 dim size

######## Padding before Convolution #######
a_pad = torch.zeros(len(icg), len(nig)+padding*2, len(njg)+padding*2).cuda()
# a_pad.size() = [64, 32+2pad, 32+2pad]
a_pad[ :, padding:padding+len(nig), padding:padding+len(njg)] = a_int.cuda()
a_pad = torch.reshape(a_pad, (a_pad.size(0), -1))  ## mergin ni and nj index into nij
# a_pad.size() = [64, (32+2pad)*(32+2pad)]

In [36]:
###########################################

p_nijg = range(a_pad.size(1)) ## paded activation's nij group [0, ...34*34-1]

a_tile = torch.zeros(n_t, n_t, tile, len(p_nijg)).cuda()
psum   = torch.zeros(n_t, n_t, tile, len(p_nijg), len(kijg)).cuda()

for ic_tile in range(n_t):
    for oc_tile in range(n_t):
        a_tile[ic_tile, oc_tile] = a_pad[ic_tile*tile:(ic_tile+1)*tile, :]
        for kij in kijg:
            # PE at row r, col c holds w_int[16*oc_tile + c, 16*ic_tile + r, kij]
            W = w_int[oc_tile*tile:(oc_tile+1)*tile, ic_tile*tile:(ic_tile+1)*tile, kij].cuda()  # [col, row]
            psum[ic_tile, oc_tile, :, :, kij] = W @ a_tile[ic_tile, oc_tile] 

# for kij in kijg:       
#     for nij in p_nijg:     # time domain, sequentially given input
#         m = nn.Linear(array_size, array_size, bias=False)
#         m.weight = torch.nn.Parameter(w_int[:,:,kij])
#         psum[:, nij, kij] = m(a_pad[:,nij]).cuda()
 

In [37]:
import math

a_pad_ni_dim = int(math.sqrt(a_pad.size(1))) # 32 + 2*pad = 34

o_ni_dim = int((a_pad_ni_dim - (ki_dim- 1) - 1)/stride + 1) #34 - 2 - 1 + 1 = 32
o_nijg = range(o_ni_dim**2) # [0, 32*32-1]    
    
psum_oc = psum.sum(dim=0).reshape(n_t*tile, len(p_nijg), len(kijg))

out = torch.zeros(len(ocg), len(o_nijg)).cuda()
  
   
### SFP accumulation ###
for o_nij in o_nijg:
    for kij in kijg:
        out[:,o_nij] = out[:,o_nij] + \
        psum_oc[:, int(o_nij/o_ni_dim)*a_pad_ni_dim + o_nij%o_ni_dim + int(kij/ki_dim)*a_pad_ni_dim + kij%ki_dim, kij]
                ## 2nd index = (int(o_nij/30)*32 + o_nij%30) + (int(kij/3)*32 + kij%3)

In [38]:
conv_ref_int = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=False).cuda()
conv_ref_int.weight = torch.nn.Parameter(weight_int.detach().cuda())
ref = conv_ref_int(a_int.unsqueeze(0).cuda()).detach().reshape(64, -1)

difference = (out - ref)    # CHANGED: compare against bias-free reference on the same integers
print("a_tile:", a_tile.shape)
print("psum:  ", psum.shape)
print("out:   ", out.shape)
print("abs diff sum:", difference.abs().sum().item(), " max:", difference.abs().max().item())

a_tile: torch.Size([4, 4, 16, 1156])
psum:   torch.Size([4, 4, 16, 1156, 9])
out:    torch.Size([64, 1024])
abs diff sum: 0.36244773864746094  max: 0.000244140625
